# DSN Mart — normalized sales target

This notebook standardizes `total_sales` **during training**, then automatically converts
predictions back to original sales units before RMSE evaluation and CSV export. It uses
`TransformedTargetRegressor` with `StandardScaler` around the existing product-history model.

Run beside the CSV files using the project Python environment. This follow-up loads the trusted
local model `outputs/product_history/pipeline.pkl`. All new files go to `outputs/target_normalized/`;
previous submissions are preserved. Run cells in order to monitor every validation fold.

Target standardization is not a way to reduce Kaggle's error by changing units. For this
ridge/spline model and its linear residual correction, affine target scaling should leave
inverse-transformed predictions essentially unchanged. The experiment checks that explicitly.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import cloudpickle
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from threadpoolctl import threadpool_limits

OUT = Path('outputs/target_normalized')
OUT.mkdir(parents=True, exist_ok=True)
train, test = pd.read_csv('train.csv'), pd.read_csv('test.csv')
sample = pd.read_csv('sample_submission.csv')
features = test.columns.drop('id').tolist()
X, y = train[features], train.total_sales
assert np.isfinite(y).all() and y.ge(0).all()
with open('outputs/product_history/pipeline.pkl', 'rb') as f:
    original = cloudpickle.load(f)
normalized = TransformedTargetRegressor(regressor=clone(original), transformer=StandardScaler())
Xd, Xh, yd, yh = train_test_split(X, y, test_size=.2, random_state=42)


## 1. Compare both models on identical validation folds
Each target scaler learns its mean and standard deviation from **training labels only**.
`predict()` automatically applies the inverse transformation. We clip negative predictions
only after converting back to sales units. Negative standardized values are valid and must
not be clipped inside the target transformation.

The original development split and fold seed are retained for comparison. These development
rows have been used in earlier experiments, so the scores are not independent final tests.


In [ ]:
predictions, records = {}, []
folds = list(KFold(5, shuffle=True, random_state=42).split(Xd))
with threadpool_limits(limits=4):
    for name, template in [('Original target', original), ('Standardized target', normalized)]:
        oof = np.zeros(len(Xd))
        for fold, (a, b) in enumerate(folds, 1):
            print(f'START {name}: fold {fold}/5', flush=True)
            fitted = clone(template).fit(Xd.iloc[a], yd.iloc[a])
            if name == 'Standardized target':
                assert np.allclose(fitted.transformer_.mean_, yd.iloc[a].mean())
                assert np.allclose(fitted.transformer_.scale_, yd.iloc[a].std(ddof=0))
                # Confirm that public predict() returns original-unit values.
                scaled = fitted.regressor_.predict(Xd.iloc[b])
                restored = fitted.transformer_.inverse_transform(np.asarray(scaled).reshape(-1, 1)).ravel()
                assert np.allclose(fitted.predict(Xd.iloc[b]), restored)
            oof[b] = np.maximum(0, fitted.predict(Xd.iloc[b]))
            rmse = float(root_mean_squared_error(yd.iloc[b], oof[b]))
            records.append({'model': name, 'fold': fold, 'rmse_original_units': rmse})
            pd.DataFrame(records).to_csv(OUT / 'fold_scores.csv', index=False)
            print(f'DONE: original-unit RMSE {rmse:.6f}', flush=True)
        predictions[name] = oof
comparison = pd.DataFrame([{'model': name,
    'rmse_original_units': root_mean_squared_error(yd, pred),
    'mae_original_units': mean_absolute_error(yd, pred)} for name, pred in predictions.items()])
comparison.to_csv(OUT / 'comparison.csv', index=False)
display(comparison)
max_difference = float(np.max(np.abs(predictions['Original target'] - predictions['Standardized target'])))
print('Maximum difference between original-unit OOF predictions:', max_difference)
print('Equivalent within numerical tolerance:', np.allclose(predictions['Original target'],
      predictions['Standardized target'], rtol=1e-8, atol=1e-6))


## 2. Previously inspected holdout diagnostic
The holdout has been inspected in prior work; we do not call it untouched or tune to it.
Only original-unit RMSE is used for the comparison. Standardized-target training is retained
as requested even if it produces equivalent predictions, not falsely described as an improvement.


In [ ]:
with threadpool_limits(limits=4):
    diagnostic = clone(normalized).fit(Xd, yd)
    hold_predictions = np.maximum(0, diagnostic.predict(Xh))
hold_rmse = float(root_mean_squared_error(yh, hold_predictions))
print('Diagnostic holdout RMSE, original sales units:', hold_rmse)


## 3. Final fit and original-unit submission
After validation, fit the target scaler and model on all labelled rows. Test data never fits
the target scaler. The exported `total_sales` values are sales, not z-scores. The scaler is saved
inside the model so subsequent predictions automatically receive the inverse transformation.


In [ ]:
print('Training final target-normalized model on all labelled rows...', flush=True)
with threadpool_limits(limits=4):
    final = clone(normalized).fit(X, y)
    pred = np.maximum(0, final.predict(test[features]))
submission = sample[['id']].copy()
submission['total_sales'] = submission.id.map(pd.Series(pred, index=test.id))
assert submission.columns.tolist() == ['id', 'total_sales']
assert submission.shape == sample.shape and submission.id.equals(sample.id)
assert submission.id.is_unique and np.isfinite(submission.total_sales).all()
assert submission.total_sales.ge(0).all()
submission.to_csv(OUT / 'submission_target_normalized.csv', index=False)
with (OUT / 'pipeline.pkl').open('wb') as f:
    cloudpickle.dump(final, f)
with (OUT / 'pipeline.pkl').open('rb') as f:
    reloaded = cloudpickle.load(f)
assert np.allclose(pred, np.maximum(0, reloaded.predict(test[features])))
saved = pd.read_csv(OUT / 'submission_target_normalized.csv')
assert saved.id.equals(sample.id) and np.allclose(saved.total_sales, submission.total_sales)
prior = pd.read_csv('outputs/product_history/submission_product_history.csv').set_index('id')
prior_aligned = prior.loc[submission.id, 'total_sales'].to_numpy()
same_as_previous = bool(np.allclose(submission.total_sales, prior_aligned, rtol=1e-8, atol=1e-6))
print('New submission equivalent to previous predictions:', same_as_previous)
display(submission.head())


## 4. Results and ready-to-paste submission description
This produces a new file to demonstrate correct target normalization, even when its predictions
match the previous model. Equivalent predictions offer no meaningful expected leaderboard gain.


In [ ]:
cv_rmse = float(comparison.loc[comparison.model.eq('Standardized target'), 'rmse_original_units'].iloc[0])
report = {'target_transformation': 'StandardScaler fitted on training labels only',
    'submission_units': 'Original sales units (inverse transformed)',
    'cv_rmse_original_units': cv_rmse,
    'previously_inspected_holdout_rmse': hold_rmse,
    'max_oof_prediction_difference': max_difference,
    'submission_equivalent_to_previous': same_as_previous,
    'full_training_target_mean': float(final.transformer_.mean_[0]),
    'full_training_target_std': float(final.transformer_.scale_[0]),
    'test_rows': len(submission)}
(OUT / 'metrics.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
description = (f'Price-spline regression with regularised product-history corrections and '
    f'training-fold target standardization. Predictions inverse-transformed to original sales units. '
    f'Five-fold CV RMSE: {cv_rmse:.2f}. Refit on all labelled data. No external data used.')
(OUT / 'submission_description.txt').write_text(description, encoding='utf-8')
display(pd.Series(report))
print('SUBMISSION:', (OUT / 'submission_target_normalized.csv').resolve())
print('DESCRIPTION:', description)
